In [8]:
import pandas as pd

df = pd.read_csv('../data/merged_data.csv')

In [9]:
print(df.isnull().sum())
print("Total rows:", df.shape[0])

SEQN           0
RIDAGEYR       0
RIAGENDR       0
BMXBMI        99
PAD680        10
PAQ605         0
PFQ061D     2766
dtype: int64
Total rows: 5533


In [10]:
# Filter age 20-65
df = df[(df['RIDAGEYR'] >= 20) & (df['RIDAGEYR'] <= 65)]

# Remove invalid sedentary time values
df = df[df['PAD680'] != 9999]

# Remove missing and invalid physical activity values
df = df[df['PAQ605'].notna()]
df = df[~df['PAQ605'].isin([7, 9])]

# Remove missing and invalid functional limitation values
df = df[df['PFQ061D'].notna()]
df = df[~df['PFQ061D'].isin([5, 7, 9])]

# Remove missing BMI
df = df[df['BMXBMI'].notna()]

# Convert sedentary time to hours and remove outliers
df['sitting_hours'] = df['PAD680'] / 60
df = df[(df['sitting_hours'] >= 0.5) & (df['sitting_hours'] <= 18)]

print("after cleaning:", df.shape)

after cleaning: (1374, 8)


In [11]:
# Binarize outcome variable: 1 = has difficulty, 0 = no difficulty
df['functional_limitation'] = (df['PFQ061D'] >= 2).astype(int)

# Binarize physical activity: 1 = active, 0 = not active
df['physically_active'] = (df['PAQ605'] == 1).astype(int)

print(df['functional_limitation'].value_counts())
print(df['physically_active'].value_counts())

functional_limitation
1    783
0    591
Name: count, dtype: int64
physically_active
0    1028
1     346
Name: count, dtype: int64


In [12]:
print("Final sample size:", df.shape[0])
print(df[['RIDAGEYR', 'BMXBMI', 'sitting_hours']].describe())

Final sample size: 1374
          RIDAGEYR       BMXBMI  sitting_hours
count  1374.000000  1374.000000    1374.000000
mean     53.039301    31.133406       5.605046
std      12.242449     8.233689       3.425961
min      20.000000    14.800000       0.500000
25%      46.000000    25.525000       3.000000
50%      59.000000    29.700000       5.000000
75%      62.000000    35.100000       8.000000
max      65.000000    74.800000      18.000000
